# RuleShift Benchmark

Attach the packaged runtime dataset and run the official benchmark task.

In [ ]:
import sys
from pathlib import Path

import kaggle_benchmarks as kbench

RUNTIME_SRC = Path("/kaggle/input/datasets/raptorengineer/ruleshift-runtime/src")
if not RUNTIME_SRC.is_dir():
    raise FileNotFoundError(
        "Runtime dataset not found at /kaggle/input/datasets/raptorengineer/ruleshift-runtime/src -- "
        "attach the raptorengineer/ruleshift-runtime dataset."
    )
if str(RUNTIME_SRC) not in sys.path:
    sys.path.insert(0, str(RUNTIME_SRC))


In [ ]:
import os

from tasks.ruleshift_benchmark.runtime import (
    PRIVATE_DATASET_ROOT_ENV_VAR,
    load_private_rows,
    load_public_rows,
    run_binary_task,
)

leaderboard_rows = load_public_rows()
PRIVATE_DATASET_ROOT = os.environ.get(PRIVATE_DATASET_ROOT_ENV_VAR)
if PRIVATE_DATASET_ROOT:
    leaderboard_rows.extend(load_private_rows(PRIVATE_DATASET_ROOT))

if not leaderboard_rows:
    raise RuntimeError("No benchmark episodes were loaded.")


## Official task

`ruleshift_benchmark_v1_binary` is the leaderboard entry point. It loops over the frozen rows, scores each episode, and returns `(numerator, denominator)`.

In [ ]:
_RULESHIFT_RESULT = None


@kbench.task(
    name="ruleshift_benchmark_v1_binary",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of marker records, "
        "then predict four post-shift probe outputs."
    ),
)
def ruleshift_benchmark_v1_binary(llm) -> tuple[int, int]:
    import json

    global _RULESHIFT_RESULT

    numerator = 0
    denominator = 0
    for row in leaderboard_rows:
        num_correct, total = run_binary_task(
            llm=llm,
            prompt_binary=row["prompt_binary"],
            probe_targets=row["probe_targets"],
        )
        numerator += int(num_correct)
        denominator += int(total)

    if denominator == 0:
        raise RuntimeError("evaluation produced a zero denominator")

    _RULESHIFT_RESULT = {
        "score": numerator / denominator,
        "numerator": numerator,
        "denominator": denominator,
        "episodes": len(leaderboard_rows),
    }

    print(json.dumps(_RULESHIFT_RESULT, indent=2))
    return (numerator, denominator)


In [ ]:
score = ruleshift_benchmark_v1_binary(kbench.llm)
result = _RULESHIFT_RESULT
if result is None:
    raise RuntimeError("ruleshift_benchmark_v1_binary did not populate _RULESHIFT_RESULT")

result


## Task selection


In [ ]:
%choose ruleshift_benchmark_v1_binary
